# 추세와 변동성 — 강의자료 정리

> **목표:** 시계열 센서 데이터를 시간 순서대로 보고, 이동평균으로 **추세**를 확인하고, 이동표준편차로 **변동성**을 확인하는 흐름을 이해한다.

---

## 0. 한눈에 보는 함수 · 메서드 · 속성

| 구분 | 사용 형태 | 역할 | 기억할 점 |
|---|---|---|---|
| 함수 | `pd.read_csv(...)` | CSV 파일을 DataFrame으로 불러오기 | 분석의 시작 |
| 함수/생성자 | `pd.Series([...])` | 1차원 Series 생성 | 이동평균 예제에서 사용 |
| 메서드 | `df.head()` | 앞부분 데이터 확인 | 값의 형식·단위 확인 |
| 속성 | `df.shape` | `(행 개수, 열 개수)` 확인 | 필터링 결과 확인에 유용 |
| 메서드 | `df.info()` | 열 이름, 자료형, 결측치 확인 | 숫자 열이 `float64` 등인지 확인 |
| 선택 | `df['열이름']` | 특정 열 선택 | 결과는 보통 Series |
| 필터링 | `df[df['unit_nr'] == 1]` | 특정 엔진만 선택 | 시계열은 엔진별로 따로 봐야 함 |
| 메서드 | `.rolling(20)` | 20개씩 묶는 이동 구간 생성 | `20`이 window 크기 |
| 메서드 | `.mean()` | 평균 계산 | 이동평균에 사용 |
| 메서드 | `.std()` | 표준편차 계산 | 이동표준편차에 사용 |
| 메서드 | `.tolist()` | Series를 list로 변환 | 예제 결과 확인에 사용 |

### 핵심 코드 흐름

```python
import pandas as pd

df = pd.read_csv("data/cmapss_fd001_sample.csv")

# 한 엔진만 선택
engine1 = df[df["unit_nr"] == 1]

# 센서 선택
sensor = engine1["s_2"]

# 이동평균
ma20 = sensor.rolling(20).mean()

# 이동표준편차
std20 = sensor.rolling(20).std()

# 평균 ± 2 × 표준편차 밴드
upper = ma20 + 2 * std20
lower = ma20 - 2 * std20
```

### 전체 분석 흐름

`데이터 불러오기 → 한 엔진 선택 → 센서 선택 → 시계열 확인 → 이동평균으로 추세 확인 → 이동표준편차로 변동성 확인 → 이상 신호 해석`


# 1. 시계열 데이터와 추세

## 1-1. 시계열 데이터란?

**시계열 데이터(Time Series)**는 **시간 순서대로 측정해 기록한 데이터**이다.

예:
- 일별 기온
- 주식 가격
- 매일 측정한 몸무게
- 설비의 온도·압력·진동 센서값

### 핵심
시계열에서 가장 중요한 것은 **순서**다.

일반 데이터는 행 순서를 바꿔도 의미가 유지되는 경우가 많지만,  
시계열 데이터는 순서를 바꾸면 **시간에 따른 변화의 흐름**이 사라진다.

---

## 1-2. 시계열의 두 축

- **가로축(X축)**: 시간, 날짜, 측정 시각, cycle
- **세로축(Y축)**: 온도, 압력, 진동 등의 센서값

따라서 선이 오른쪽으로 갈수록 올라가면  
**시간이 지나면서 값이 증가한다**고 해석할 수 있다.

### 그래프를 볼 때 주의
반드시 먼저 확인할 것:
1. 가로축의 단위
2. 세로축의 범위

같은 데이터도 세로축 범위에 따라 매우 평평하거나 급격하게 상승하는 것처럼 보일 수 있다.


## 1-3. C-MAPSS 데이터 구조

강의에서는 NASA의 항공 엔진 데이터인 **C-MAPSS** 예제를 사용한다.

대표 열:

| 열 이름 | 의미 |
|---|---|
| `unit_nr` | 엔진 번호 |
| `time_cycles` | 가동 주기 |
| `s_2` | 센서 값 예시 |

날짜가 없어도 `1 → 2 → 3 → ...`처럼 순서가 명확한 `time_cycles`가 있으면 시간축으로 사용할 수 있다.

### 왜 한 엔진씩 골라야 할까?

엔진 1의 마지막 cycle 뒤에 엔진 2의 첫 cycle이 붙으면 시간 흐름이 끊긴다.  
따라서 시계열 분석에서는 먼저 **한 엔진만 선택**해야 한다.


In [ ]:
import pandas as pd

# 강의자료 예시
df = pd.read_csv("data/cmapss_fd001_sample.csv")

print(df.shape)

engine1 = df[df["unit_nr"] == 1]
print(engine1.shape)

s2 = engine1["s_2"]


## 1-4. 불러온 직후 확인할 3가지

### `head()`
처음 몇 행을 확인해 값의 모양, 단위, 형식을 살펴본다.

### `shape`
행과 열의 개수를 확인한다.

```python
df.shape
```

예: `(1031, 10)` → 1031행, 10열

### `info()`
각 열의 자료형을 확인한다.

센서값이 숫자처럼 보여도 문자열로 읽히면 평균·표준편차 계산이 어려울 수 있다.  
따라서 분석 전에 자료형을 확인하는 습관이 중요하다.


In [ ]:
# 분석 전 기본 점검
df.head()
df.shape
df.info()


# 2. 추세와 이동평균

## 2-1. 노이즈(Noise)

센서값이 작은 폭으로 위아래로 떨리는 현상을 **노이즈(잡음)**라고 한다.

원인 예:
- 측정 오차
- 순간적인 조건 변화
- 외부 요인

현실의 센서 데이터에는 노이즈가 존재한다.

### 핵심
목표는 노이즈를 완전히 없애는 것이 아니라,  
**노이즈에 가려진 실제 변화의 방향을 찾는 것**이다.


## 2-2. 추세(Trend)

**추세**는 작은 흔들림을 무시하고 길게 봤을 때 나타나는 **데이터의 전체 방향**이다.

- 상승 추세: `↗`
- 하강 추세: `↘`
- 평탄: `→`

### 쉬운 구분

개별 값은 계속 오르내릴 수 있다.

`100 → 102 → 99 → 103 → 101 → 105`

하지만 전체적으로 보면 `100 → 105`이므로 **상승 추세**가 있다고 볼 수 있다.

즉,

> **개별 값의 변화 ≠ 전체 추세**


## 2-3. 이동평균(Moving Average)

이동평균은 **일정 개수의 값을 묶어 평균을 낸 뒤, 그 구간을 한 칸씩 이동하면서 계속 평균을 계산하는 방법**이다.

### window란?
`window` = **한 번에 묶어서 평균을 낼 값의 개수**

예를 들어 데이터가

`[10, 12, 15, 14, 18]`

이고 `window = 3`이면:

1. `(10 + 12 + 15) / 3 = 12.33`
2. `(12 + 15 + 14) / 3 = 13.67`
3. `(15 + 14 + 18) / 3 = 15.67`

결과:

`[12.33, 13.67, 15.67]`

이동평균은 데이터 위에 작은 창문(window)을 놓고 한 칸씩 옮겨가는 것으로 생각하면 이해하기 쉽다.


In [ ]:
import pandas as pd

s = pd.Series([10, 12, 15, 14, 18])

moving_avg = s.rolling(window=3).mean()

print(moving_avg.tolist())
# [nan, nan, 12.33..., 13.66..., 15.66...]


### 앞부분에 `NaN`이 생기는 이유

`window=3`이면 평균을 계산하려면 최소 3개의 값이 필요하다.

- 첫 번째 위치: 값 1개 → 계산 불가
- 두 번째 위치: 값 2개 → 계산 불가
- 세 번째 위치: 값 3개 → 계산 가능

따라서 앞부분의 `NaN`은 **오류가 아니라 정상적인 결과**다.


## 2-4. 이동평균이 추세를 드러내는 이유

노이즈는 위아래로 흔들리기 때문에 평균을 계산하면 서로 상쇄되는 경향이 있다.

예:

`100, 102, 98`

평균은 `100`

즉 `+2`와 `-2`가 평균 속에서 서로 상쇄된다.

반대로 한 방향의 변화는 평균을 내도 남는다.

이처럼 데이터를 부드럽게 다듬어 큰 흐름을 보기 쉽게 만드는 것을 **평활화(smoothing)**라고 한다.

### 핵심 한 문장

> **노이즈는 평균 속에서 줄어들고, 추세는 남는다.**


## 2-5. window 크기의 의미

| 구분 | 작은 window | 큰 window |
|---|---|---|
| 선 모양 | 비교적 거침 | 더 매끈함 |
| 변화 반응 | 빠름 | 느림 |
| 앞부분 NaN | 적음 | 많음 |
| 장점 | 빠른 변화 확인 | 큰 흐름 확인 |

### 해석
- 작은 window: 변화에 민감하지만 노이즈가 더 남는다.
- 큰 window: 더 부드럽지만 변화에 둔감하다.

따라서 **큰 window가 무조건 좋은 것은 아니다.**

강의 예제의 192주기 데이터에서는 `10 ~ 30` 정도를 적정 window 예시로 제시하지만,  
정답이 정해져 있는 것은 아니며 분석 목적에 맞게 선택해야 한다.


In [ ]:
# 같은 센서에서 window 크기 비교 예시
ma5 = s.rolling(5).mean()
ma20 = s.rolling(20).mean()
ma50 = s.rolling(50).mean()


## 2-6. 추세의 방향과 해석은 다르다

데이터가 말해주는 사실:

> "온도가 상승하고 있다."

설비 지식이 필요한 해석:

> "온도가 상승해서 위험하다."

같은 상승 추세라도 센서마다 의미가 다를 수 있다.

따라서:
1. 데이터로 방향을 확인하고
2. 센서의 정상 방향과 비교하고
3. 변화 속도까지 함께 본다.


# 3. 변동성 분석

## 3-1. 왜 추세만 보면 부족할까?

두 설비의 평균이 같아도 상태는 다를 수 있다.

### 안정적인 예
`99, 100, 101, 100, 99`

### 불안정한 예
`80, 120, 85, 115, 90`

두 데이터 모두 평균은 비슷할 수 있지만, 두 번째 데이터가 훨씬 크게 흔들린다.

고장은 평균값의 변화보다 **흔들림의 증가**, 즉 변동성 증가로 먼저 신호를 보낼 수 있다.


## 3-2. 변동성(Volatility)

**변동성**은 값이 평균을 중심으로 **얼마나 크게 흔들리는지**를 의미한다.

- 변동성 작음 → 평균 근처에서 안정적
- 변동성 큼 → 평균에서 멀리 위아래로 크게 움직임

### 가장 쉬운 구분

> **추세 = 어디로 가는가?**  
> **변동성 = 얼마나 불안한가?**


## 3-3. 표준편차(Standard Deviation)

표준편차는 값들이 평균에서 얼마나 흩어져 있는지를 나타내는 값이다.

예:

| 데이터 | 표준편차 해석 |
|---|---|
| `[10, 10, 10, 10]` | `std = 0`, 전혀 흔들리지 않음 |
| `[8, 12, 9, 11]` | 조금 흩어짐 |
| `[2, 18, 5, 15]` | 크게 흩어짐 |

### 장점
표준편차는 원래 값과 단위가 같다.

예를 들어 온도 데이터에서 표준편차가 `2`라면  
평균에서 대략 **2도 정도 흔들리는 정도**로 직관적으로 해석할 수 있다.


## 3-4. 이동표준편차

이동표준편차는 이동평균과 원리가 거의 같다.

차이는 마지막 계산만 다르다.

```python
# 이동평균
sensor.rolling(20).mean()

# 이동표준편차
sensor.rolling(20).std()
```

### 의미
- 이동평균: 시간이 지나면서 **평균이 어떻게 변하는지**
- 이동표준편차: 시간이 지나면서 **흔들림이 어떻게 변하는지**

따라서 이동표준편차가 점점 커지면  
설비 상태가 점점 불안정해지는 신호로 볼 수 있다.


In [ ]:
# 강의자료 예시
std20 = engine1["s_3"].rolling(20).std()

# 강의자료에서는 예시로
# 초반 구간 약 0.931
# 후반 구간 약 1.862
# → 후반 변동성이 약 2배 증가하는 사례를 제시


## 3-5. 변동성 증가가 설비에 갖는 의미

정상 상태에서는 센서값이 일정한 범위 안에서 안정적으로 움직인다.

문제가 시작되면 값이 불규칙해지면서 변동성이 커질 수 있다.

강의자료 예:
- 베어링 마모 → 진동이 들쭉날쭉
- 부품 헐거움 → 압력이 불안정

### 핵심
평균이 아직 크게 변하지 않았더라도  
**변동성은 먼저 증가할 수 있다.**

그래서 평균과 변동성을 따로 본다.


## 3-6. 평균 ± 표준편차 밴드

이동평균선 주변에 표준편차를 이용한 범위를 그리면  
추세와 변동성을 한 번에 확인할 수 있다.

강의자료의 구성:

`MA ± 2 × STD`

즉:

```python
upper = ma20 + 2 * std20
lower = ma20 - 2 * std20
```

### 해석
- 가운데 선 → **추세**
- 밴드의 폭 → **변동성**
- 밴드가 넓어짐 → 더 불안정한 구간


In [ ]:
sensor = engine1["s_2"]

ma20 = sensor.rolling(20).mean()
std20 = sensor.rolling(20).std()

upper = ma20 + 2 * std20
lower = ma20 - 2 * std20


## 3-7. 변동성 패턴: 점증과 급증

### ① 점증
변동성이 서서히 증가하는 패턴.

강의자료에서는:
- 점진적 마모
- 노화
- 점점 불안정해지는 상태

와 연결해 설명한다.

→ 정비 계획을 미리 세울 때 참고할 수 있다.

### ② 급증
어느 시점에서 변동성이 갑자기 크게 튀는 패턴.

하지만:

> **급증 = 무조건 고장**은 아니다.

원본 센서값과 운영 기록을 함께 확인해 원인을 판단해야 한다.


# 4. 강의 실습 한눈에 정리

| 실습 | 목표 | 핵심 개념 |
|---|---|---|
| 실습 1 | 한 엔진의 센서 시계열 그리기 | 시계열 |
| 실습 2 | 이동평균선으로 추세 보기 | `rolling().mean()` |
| 실습 3 | window 5·20·50 비교 | window 크기 |
| 실습 4 | 여러 센서 추세 방향 비교 | 상승·하강·평탄 |
| 실습 5 | 이동표준편차로 변동성 구간 찾기 | `rolling().std()` |
| 실습 6 | 평균과 변동성 밴드 함께 보기 | `MA ± 2×STD` |
| 실습 7 | 변동성 패턴 해석 | 점증·급증 |


# 5. 시험·복습용 핵심 요약

## 반드시 구분

| 개념 | 질문 | 대표 계산 |
|---|---|---|
| 시계열 | 시간에 따라 어떻게 변하는가? | 시간 순서 유지 |
| 추세 | 어디로 가는가? | 이동평균 |
| 변동성 | 얼마나 흔들리는가? | 표준편차 |
| 이동평균 | 구간 평균이 시간에 따라 어떻게 변하는가? | `.rolling().mean()` |
| 이동표준편차 | 구간 흔들림이 시간에 따라 어떻게 변하는가? | `.rolling().std()` |

## 꼭 기억할 포인트

1. 시계열은 **순서가 핵심**이다.
2. 여러 엔진을 섞지 말고 **한 엔진씩 분석**한다.
3. 데이터 불러온 직후 `head()`, `shape`, `info()`를 확인한다.
4. 이동평균은 노이즈를 줄여 **추세**를 보기 쉽게 한다.
5. window가 크면 더 매끈하지만 변화에 둔감하다.
6. 표준편차는 **흩어진 정도 = 변동성**을 나타낸다.
7. 이동표준편차는 시간에 따라 변동성이 변하는 모습을 추적한다.
8. 평균이 그대로여도 변동성이 커지면 이상 신호가 될 수 있다.
9. `MA ± 2 × STD` 밴드에서 가운데 선은 추세, 폭은 변동성을 보여준다.
10. 데이터가 말하는 **사실**과 설비 지식에 기반한 **해석**은 구분해야 한다.

---

## 한 문장으로 끝내기

> **센서값을 시간 순서대로 보고, 이동평균으로 큰 방향을 찾고, 이동표준편차로 흔들림을 확인하여 설비 이상 신호를 해석한다.**
